# Clearing out the old copy

The course repo was rebuilt from scratch. Copies made **before** the rebuild still hold old files, and one of them, an old cached copy of the dataset, will stop session 2 from running. This notebook clears them out.

**About five minutes. You don't write any code:** you run one cell and read what it prints.

**What it removes**
- the course's cached dataset files in your workspace volume (`/Volumes/workspace/default/lunch_and_learn`), which are rebuilt in seconds
- the table session 5 saved for you (`hero_offenders_clean_` plus your name), and its saved history

**What it never touches:** anything else in that volume, your other tables, and every notebook.

## Step 1 · By hand, before you run anything (2 minutes)

1. **Keep your own answers.** If you typed exercise answers in the old copy and want to keep them, copy just those cells into a note now.
2. **Delete the old copy.** In the left sidebar click **Workspace**, find the old `lunch-and-learn-data-cleansing` folder, right-click it and choose **Move to Trash**. Then click **Trash** in the left sidebar and choose **Empty Trash**: Databricks otherwise keeps deleted folders for 30 days.
3. **Get the new copy.** **Workspace → Create → Git folder**, paste `https://github.com/pugwala/lunch-and-learn-data-cleansing`, and click **Create**.
4. **Open this notebook from the new copy:** `notebooks` → `clearing-out-the-old`. If you are reading this inside the old copy, stop here, do step 2, and open it again from the new one.

## Step 2 · Run the cell below

Click anywhere in the gray cell and press **Shift + Enter**. It prints a short checklist:

- ✅ means done
- ⚠️ means something for you to do; it says what

Running it again is safe: anything already cleared stays cleared.

In [ ]:
# ▶ Run this cell (Shift + Enter). You don't need to change anything here.
# It clears out the old copy's data, checks for leftover copies of the course, and builds fresh data for session 2.
import shutil
import sys
from pathlib import Path


def _find_course_files():
    """The course's Git folder: where this notebook's lnl/ package lives."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")


COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, env

to_do = []


def done(text):
    """Report something finished."""
    print("✅", text)


def needs(text):
    """Report something the reader still has to do."""
    print("⚠️ ", text)
    to_do.append(text)


def is_old(item):
    """True for course data written before the rebuild: its record key isn't offender_number."""
    try:
        if item.is_dir():   # session 5's saved output, outside Databricks
            import pandas as pd
            return "offender_number" not in pd.read_parquet(item).columns
        with item.open(errors="replace") as f:
            return f.readline().split(",")[0].strip() != "offender_number"
    except Exception:
        return True


# 1. The cached dataset. Its file name didn't change in the rebuild, so an old one would be reused and break session 2.
if env.IS_DATABRICKS and config.DATABRICKS_VOLUME and not Path(config.DATABRICKS_VOLUME).exists():
    needs(f"The course volume {config.DATABRICKS_VOLUME} is missing. Ask your instructor to create it "
          "(Catalog > workspace > default > Create > Volume, named lunch_and_learn).")
folder = env.data_dir()
current = {env.paths()["data"].name, env.paths()["answer_key"].name}   # this copy's dataset files: kept if they're new
old = ([p for p in sorted(folder.glob("hero_offender_data_*")) if p.name not in current or is_old(p)]
       + [p for p in sorted(folder.glob("hero_offenders_clean_*")) if is_old(p)])
for item in old:
    shutil.rmtree(item) if item.is_dir() else item.unlink()
done(f"Removed {len(old)} old course data file(s) from {folder}" if old
     else f"No old course data in {folder} (anything from the new copy is kept)")

# 2. The table session 5 saved for you (Databricks only). Dropping it removes its saved history too.
spark = globals().get("spark")
if env.IS_DATABRICKS and spark is not None and config.DATABRICKS_CATALOG and config.DATABRICKS_SCHEMA:
    user = spark.sql("SELECT current_user()").first()[0].split("@")[0]
    user = "".join(c if c.isalnum() else "_" for c in user).lower()
    table = f"{config.DATABRICKS_CATALOG}.{config.DATABRICKS_SCHEMA}.hero_offenders_clean_{user}"
    if spark.catalog.tableExists(table) and "offender_number" in spark.table(table).columns:
        done(f"Your session 5 table {table} is already from the new copy: kept")
    elif spark.catalog.tableExists(table):
        spark.sql(f"DROP TABLE {table}")
        done(f"Dropped your old session 5 table {table} (Databricks erases it for good within 7 days)")
    else:
        done(f"No session 5 table to drop ({table})")
else:
    done("Not on Databricks: there is no session 5 table to drop")

# 3. Any other copy of the course still in the workspace (an old Git folder).
if env.IS_DATABRICKS:
    try:
        from databricks.sdk import WorkspaceClient
        here = str(COURSE).removeprefix("/Workspace")
        others = [r.path for r in WorkspaceClient().repos.list()
                  if "lunch-and-learn-data-cleansing" in (r.url or "") and (r.path or "").removeprefix("/Workspace") != here]
        for path in others:
            needs(f"Another copy of the course is still at {path}. Right-click it in Workspace, choose Move to Trash, "
                  "then empty Trash, and run this cell again.")
        if not others:
            done("No other copy of the course in your workspace")
    except Exception:
        needs("Couldn't list your Git folders. Look in Workspace for any other lunch-and-learn-data-cleansing "
              "folder, and move it to Trash.")
    print("   Remember to empty Trash too: deleted folders stay there for 30 days.")

# 4. Fresh data, so session 2 starts straight away, and a check that it's the new dataset.
paths = env.generate()
header = paths["data"].open().readline().strip().split(",")[0]
if header == "offender_number":
    done(f"Fresh dataset built: {config.ROWS:,} records, the same for everyone")
else:
    needs("The dataset doesn't look like the new one. Make sure this notebook is open from the new copy, "
          "then run this cell again.")

print()
print("All clear. Open notebooks > 02_cleansing and run its first cell." if not to_do
      else f"{len(to_do)} thing(s) to do above (⚠️), then run this cell again.")

## Step 3 · You're done

When the last line says **All clear**, open `notebooks` → `02_cleansing` and run its first cell as usual.

**Also delete, by hand:** any course notebooks you downloaded, the old student handout PDF, and any fork of the repo you made on GitHub.

**If a line shows ⚠️**

- *Another copy of the course is still at …*: right-click that folder in **Workspace**, choose **Move to Trash**, empty **Trash**, and run the cell again.
- *The course volume … is missing*: ask your instructor; it's a one-time setup.
- *Can't find the course files*: open this notebook from inside the course's Git folder, not from a copy somewhere else.
- *The dataset doesn't look like the new one*: you're running this from the old copy. Go back to Step 1.